# 06 — TabFM vs. tree models: next-week red tide risk

## Objective
Test Google's **TabFM** (a pre-trained "zero-shot" foundation model for tables) against the models the team already
uses, on the same task, data, split and scores:

> Predict next week's *K. brevis* class at each sampling site (`y_class`: **0** < 5,000, **1** = 5,000–99,999,
> **2** ≥ 100,000 cells/L) from what we know this week.

### What TabFM is
TabFM is not trained on our data. `fit()` only stores the training rows; at prediction time the model reads them as
*context* (in-context learning, like a prompt for an LLM) and predicts the new rows in one forward pass. It was
pre-trained on hundreds of millions of synthetic tables. Google reports it beats tuned XGBoost / CatBoost / LightGBM on
TabArena (38 classification datasets of 700–150,000 rows), so it is worth checking on ours.

### What this notebook runs
| Section | Question |
|---|---|
| 3. Validation benchmark | Train ≤ 2023 → predict 2024. Which model scores best? |
| 4. Learning curve | Does TabFM win when there is little data, and do trees catch up with more? |
| 5. Feature-set sweep | Does any model use the environmental candidates better? |
| 6. Drivers | Which features each model relies on |
| 7. Regional table | The 364-week regional table, TabFM's small-data home ground |
| 8. Final check | Train ≤ 2024 → predict 2025 (once), and false alarms in 2026 |
| 9. Verdict | Printed automatically from the numbers above |

### How to run it in Colab
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. **Runtime → Run all.** The first cell installs TabFM and CatBoost; the data is downloaded from the GitHub repo.
3. Expect roughly 15–30 minutes on a T4, most of it TabFM. Switches in the **Settings** cell turn sections off.

Every model gets the **same rows, features, split and decision rule**, so the leaderboards are comparable.

## 0. Setup

In [ ]:
# Colab already has pandas, scikit-learn, PyTorch, LightGBM and XGBoost. This adds TabFM (PyTorch backend) and CatBoost.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabfm[pytorch]", "catboost"], check=True)
else:
    print('Not in Colab: install the packages yourself if needed: pip install "tabfm[pytorch]" catboost lightgbm xgboost')

In [ ]:
import json
import os
import time
import traceback
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix,
                             roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb

try:
    import catboost
except ImportError:
    catboost = None

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch, HAS_GPU = None, False

try:
    import tabfm
    from tabfm import TabFMClassifier
    HAS_TABFM = hasattr(tabfm, "tabfm_v1_0_0_pytorch")
except ImportError:
    HAS_TABFM = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (TabFM will be slow on CPU)")
print("TabFM installed:", HAS_TABFM, "| CatBoost installed:", catboost is not None)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| torch", torch.__version__ if torch else "-")

### Settings
Change these and re-run. `SCORE_TEST` controls section 8: the project rule is to look at 2025 only for the final
comparison, so leave it off while you are still changing models or features.

In [ ]:
SEED = 42

FEATURE_SET = "core"            # "core", "core+temperature", "core+chlorophyll", "core+wind", "core+all"
DECISION_RULE = "balanced"      # "balanced" (divide by the training class shares, like class_weight="balanced") or "argmax"

RUN_TABFM = True                # TabFM, default settings (32 ensemble members)
RUN_TABFM_ENSEMBLE = True       # TabFM.ensemble(): feature crosses + SVD features + calibration (slower)
TABFM_BATCH_SIZE = 1            # ensemble members per forward pass; raise to 4-8 on a large GPU to go faster

TUNE_TREES = True               # random search for LightGBM and XGBoost on the 2024 validation year
N_TRIALS = 30                   # random-search trials per model

RUN_LEARNING_CURVE = True       # section 4
RUN_FEATURE_SWEEP = True        # section 5
RUN_IMPORTANCE = True           # section 6 (trees); TabFM importance is much slower:
RUN_TABFM_IMPORTANCE = False
RUN_REGIONAL = True             # section 7
SCORE_TEST = True               # section 8: refit on <= 2024 and score 2025 + 2026

N_BOOT = 1000                   # bootstrap resamples for the confidence intervals
OUTPUT_DIR = "outputs_06"       # leaderboards are saved here as CSV
os.makedirs(OUTPUT_DIR, exist_ok=True)

if RUN_TABFM and not HAS_TABFM:
    print("TabFM is not installed, so the TabFM rows will be skipped.")

### Load the final modelling tables
The notebook looks for `data/final/` in the repo first (running locally), then in the team Drive folder, and otherwise
downloads the files from GitHub. `feature_sets.json` is stored with Git LFS, so it comes from the LFS download URL.

In [ ]:
REPO = "mhosigiri/Biointerphase-1B-forecasting-harmful-algal-bloom-risk"
BRANCH = "main"
FILES = ["site_week_table_fe.parquet", "model_table_weekly_fe_reduced.parquet", "feature_sets.json"]
RAW_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/final/"
LFS_URL = f"https://media.githubusercontent.com/media/{REPO}/{BRANCH}/data/final/"


def is_lfs_pointer(path):
    with open(path, "rb") as f:
        return f.read(40).startswith(b"version https://git-lfs")


def find_data_dir():
    candidates = [os.path.join("..", "data", "final"), os.path.join("data", "final"),
                  "/content/drive/MyDrive/biointerphase_data/final"]
    for folder in candidates:
        if all(os.path.exists(os.path.join(folder, name)) and not is_lfs_pointer(os.path.join(folder, name))
               for name in FILES):
            return folder
    folder = "data_final"
    os.makedirs(folder, exist_ok=True)
    for name in FILES:
        path = os.path.join(folder, name)
        if not os.path.exists(path) or is_lfs_pointer(path):
            urllib.request.urlretrieve(RAW_URL + name, path)
            if is_lfs_pointer(path):
                urllib.request.urlretrieve(LFS_URL + name, path)
    return folder


DATA_DIR = find_data_dir()
site = pd.read_parquet(os.path.join(DATA_DIR, "site_week_table_fe.parquet"))
regional = pd.read_parquet(os.path.join(DATA_DIR, "model_table_weekly_fe_reduced.parquet"))
with open(os.path.join(DATA_DIR, "feature_sets.json")) as f:
    feature_sets = json.load(f)

print("Data folder:", os.path.abspath(DATA_DIR))
print("Site-week table:", site.shape, "| regional table:", regional.shape)

In [ ]:
# Keep labelled rows only: y_class is NaN when the next week was not sampled
site = site[site["y_class"].notna()].copy()
site["y_class"] = site["y_class"].astype(int)
regional = regional[regional["y_class"].notna()].copy()
regional["y_class"] = regional["y_class"].astype(int)

CLASSES = [0, 1, 2]
CLASS_NAMES = ["0 no bloom", "1 at risk", "2 bloom"]
SPLITS = ["train", "validate", "test", "holdout"]

counts = pd.crosstab(site["split"], site["y_class"]).reindex(SPLITS)
counts.columns = CLASS_NAMES
onsets = site.assign(onset=(site["site_last_class"] == 0) & (site["y_class"] > 0)).groupby("split")["onset"].sum()
summary = counts.assign(total=counts.sum(axis=1), onsets=onsets.reindex(SPLITS).astype(int))
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
counts.plot.bar(stacked=True, ax=axes[0], color=["#9ecae1", "#fdae6b", "#de2d26"], rot=0)
axes[0].set_title("Site-week rows per split and class")
axes[0].set_ylabel("rows")
(counts.div(counts.sum(axis=1), axis=0) * 100).plot.bar(stacked=True, ax=axes[1], color=["#9ecae1", "#fdae6b", "#de2d26"],
                                                   rot=0, legend=False)
axes[1].set_title("Class shares (%): blooms get rarer after 2023")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

### Features
`site_core` is the 12-feature set chosen in notebook 05 (bloom history at the site, its neighbours and the region,
bloom duration, season). The candidate groups add environmental features. Columns under `do_not_use` are never used.

In [ ]:
def site_features(name):
    core = list(feature_sets["site_core"])
    groups = feature_sets["site_candidates"]
    if name == "core":
        return core
    if name == "core+all":
        return core + [c for g in ["temperature", "chlorophyll", "wind"] for c in groups[g]]
    return core + list(groups[name.split("+", 1)[1]])


FEATURES = site_features(FEATURE_SET)
LAST_CLASS = "site_last_class"      # persistence baseline and onset definition

train = site[site["split"] == "train"]
val = site[site["split"] == "validate"]
print(f"Feature set '{FEATURE_SET}': {len(FEATURES)} features")
print(FEATURES)
print(f"Train rows: {len(train):,} | validation rows: {len(val):,}")
display(train[FEATURES].isna().mean().rename("missing share (train)").to_frame().T.style.format("{:.1%}"))

## 1. How every model is scored

Every model outputs three probabilities per row. Scores are computed from those probabilities in the same way:

| Score | Meaning | Why |
|---|---|---|
| **Macro-F1** | F1 averaged over the three classes | Main project metric; not fooled by the 90 % "no bloom" rows |
| Macro-F1 (argmax) | Same, with the plain most-likely class | Shows how much the decision rule matters |
| Balanced accuracy | Mean recall of the three classes | |
| Recall 1 / Recall 2 | Share of real at-risk / bloom rows caught | "Ability to identify at-risk conditions" |
| **Onset recall** | New blooms caught (site was 0 last time, ≥ 5k next week) | The only weeks where copying last week cannot help |
| Missed blooms | Real class 2 predicted as 0 | The costliest error |
| False alarms | Real class 0 predicted as 1 or 2 | Cost of extra sampling |
| **AUC ≥5k** | ROC-AUC of P(class ≥ 1) | Ranking quality, independent of any threshold |
| AP ≥5k | Average precision of P(class ≥ 1) | Same, focused on the rare positives |
| **Onset AUC** | AUC ≥5k on onset-eligible rows only | Can the model see a new bloom coming? |
| Log loss | Quality of the probabilities | Lower is better; used to tune the trees |

**Decision rule.** No model is trained with class weights. Instead, every model's probabilities are divided by the
training class shares before taking the most likely class (`DECISION_RULE = "balanced"`). This is what
`class_weight="balanced"` does, but it applies equally to TabFM, which has no class-weight option. The threshold-free
scores (AUC, AP, log loss) do not depend on this choice.

**Uncertainty.** 2024 has only 123 at-risk and 45 bloom rows. Confidence intervals come from a bootstrap that resamples
whole **weeks** (sites in the same week are not independent). Differences inside the intervals are noise.

In [ ]:
def align_proba(classes, proba):
    # Put the probability columns in the order 0, 1, 2 (a class missing from training gets probability 0)
    proba = np.asarray(proba, dtype=float)
    out = np.zeros((len(proba), len(CLASSES)))
    for j, c in enumerate(np.asarray(classes).astype(int)):
        out[:, CLASSES.index(c)] = proba[:, j]
    return out / out.sum(1, keepdims=True).clip(1e-12)


def class_prior(y):
    return np.bincount(np.asarray(y, dtype=int), minlength=len(CLASSES)) / len(y)


def decide(proba, prior):
    if DECISION_RULE == "balanced":
        return np.argmax(proba / np.clip(prior, 1e-6, None), axis=1)
    return np.argmax(proba, axis=1)


def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan


def score(y, proba, last_class, prior):
    y = np.asarray(y, dtype=int)
    last = np.asarray(last_class, dtype=float)
    pred = decide(proba, prior)
    p_any = proba[:, 1] + proba[:, 2]
    onset_rows = last == 0
    onset = onset_rows & (y > 0)
    cm = confusion_matrix(y, pred, labels=CLASSES)
    return {
        "macro_f1": f1_score(y, pred, average="macro", labels=CLASSES, zero_division=0),
        "macro_f1_argmax": f1_score(y, proba.argmax(axis=1), average="macro", labels=CLASSES, zero_division=0),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "recall_1": recall_score(y, pred, labels=[1], average="macro", zero_division=0),
        "recall_2": recall_score(y, pred, labels=[2], average="macro", zero_division=0),
        "precision_2": precision_score(y, pred, labels=[2], average="macro", zero_division=0),
        "onset_recall": (pred[onset] > 0).mean() if onset.any() else np.nan,
        "missed_blooms": int(cm[2, 0]),
        "false_alarms": int(cm[0, 1:].sum()),
        "auc_any": safe_auc(y > 0, p_any),
        "ap_any": average_precision_score(y > 0, p_any) if (y > 0).any() else np.nan,
        "onset_auc": safe_auc(y[onset_rows] > 0, p_any[onset_rows]),
        "log_loss": log_loss(y, np.clip(proba, 1e-7, 1), labels=CLASSES),
    }


HIGHER_IS_BETTER = {"macro_f1": True, "macro_f1_argmax": True, "balanced_acc": True, "recall_1": True, "recall_2": True, "precision_2": True,
                    "onset_recall": True, "missed_blooms": False, "false_alarms": False, "auc_any": True,
                    "ap_any": True, "onset_auc": True, "log_loss": False}
RANK_METRICS = ["macro_f1", "auc_any", "ap_any", "onset_auc", "recall_2", "log_loss"]


def week_bootstrap(y, proba_by_model, groups, last_class, prior, metrics=("macro_f1", "auc_any", "onset_auc"),
                   n_boot=N_BOOT):
    # Resample whole weeks with replacement; returns {metric: DataFrame(boot x model)}
    rng = np.random.default_rng(SEED)
    y = np.asarray(y, dtype=int)
    last_class = np.asarray(last_class, dtype=float)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    rows_of = {g: np.flatnonzero(groups == g) for g in uniq}
    preds = {m: decide(p, prior) for m, p in proba_by_model.items()}
    out = {k: [] for k in metrics}
    for _ in range(n_boot):
        idx = np.concatenate([rows_of[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        yb = y[idx]
        for k in metrics:
            row = {}
            for m, p in proba_by_model.items():
                if k == "macro_f1":
                    row[m] = f1_score(yb, preds[m][idx], average="macro", labels=CLASSES, zero_division=0)
                elif k == "auc_any":
                    row[m] = safe_auc(yb > 0, p[idx, 1] + p[idx, 2])
                elif k == "onset_auc":
                    rows = last_class[idx] == 0
                    row[m] = safe_auc(yb[rows] > 0, (p[idx, 1] + p[idx, 2])[rows])
            out[k].append(row)
    return {k: pd.DataFrame(v) for k, v in out.items()}

## 2. The models

| Model | Type | Settings |
|---|---|---|
| Majority class | baseline | always the training class shares |
| Persistence | baseline | next week = the site's last observed class (`site_last_class`) |
| Logistic regression | linear | median imputation + missing flags, scaled, multinomial |
| Random forest | bagged trees | 500 trees, `min_samples_leaf=10` (as in `random_forest.ipynb`), NaN handled natively |
| HistGradientBoosting | boosted trees | scikit-learn defaults |
| LightGBM / XGBoost (tuned) | boosted trees | random search on 2024 with early stopping (`TUNE_TREES`) |
| CatBoost | boosted trees | 1,000 iterations, depth 6 |
| **TabFM** | foundation model | default: 32 ensemble members, all training rows as context |
| **TabFM ensemble** | foundation model | `TabFMClassifier.ensemble()`: feature crosses, SVD features, NNLS weights, calibration |

TabFM fills missing numbers with the column mean, so it also gets a 0/1 "was missing" column for every feature that
has gaps (the trees see the NaN directly). The README says the default context is 100 rows, but the code's default is
`max_num_rows=None`, which uses **all** training rows; it is set explicitly below.

In [ ]:
def with_missing_flags(X, cols):
    X = X.astype(float).copy()
    for c in cols:
        X[c + "_missing"] = X[c].isna().astype(float)
    return X


class Persistence:
    # Copies the last observed class; probability 1 on that class
    def __init__(self, column):
        self.column = column
        self.classes_ = np.array(CLASSES)

    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        last = X[self.column].fillna(0).astype(int).values
        return np.eye(len(CLASSES))[last]


class Majority:
    def fit(self, X, y):
        self.classes_ = np.array(CLASSES)
        self.prior_ = class_prior(y)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


class TabFMWrapper:
    # TabFM with missing-value flags; one shared pre-trained model is reused by every instance
    _model = None

    def __init__(self, preset="default", n_estimators=32, cache_context=False):
        self.preset, self.n_estimators, self.cache_context = preset, n_estimators, cache_context

    @classmethod
    def load_model(cls):
        if cls._model is None:
            device = "cuda" if HAS_GPU else "cpu"
            # bfloat16 is the design default; T4-class GPUs (compute < 8.0) have no fast bf16, so use float32 there
            fast_bf16 = HAS_GPU and torch.cuda.get_device_capability(0)[0] >= 8
            dtype = torch.bfloat16 if fast_bf16 else torch.float32
            t0 = time.time()
            cls._model = tabfm.tabfm_v1_0_0_pytorch.load(model_type="classification", device=device, dtype=dtype)
            print(f"TabFM weights loaded on {device} ({dtype}) in {time.time() - t0:.0f} s")
        return cls._model

    def fit(self, X, y):
        model = self.load_model()
        self.flag_cols_ = [c for c in X.columns if X[c].isna().any()]
        kwargs = dict(max_num_rows=None, batch_size=TABFM_BATCH_SIZE, random_state=SEED,
                      cache_context=self.cache_context)
        if self.preset == "ensemble":
            self.clf_ = TabFMClassifier.ensemble(model, **kwargs)
        else:
            self.clf_ = TabFMClassifier(model=model, n_estimators=self.n_estimators, **kwargs)
        self.clf_.fit(with_missing_flags(X, self.flag_cols_), np.asarray(y, dtype=int))
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, X):
        proba = np.asarray(self.clf_.predict_proba(with_missing_flags(X, self.flag_cols_)))
        if HAS_GPU:
            torch.cuda.empty_cache()
        return proba


def logistic():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(C=1.0, max_iter=5000))


def random_forest():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=10, n_jobs=-1, random_state=SEED)


def hist_gb():
    return HistGradientBoostingClassifier(random_state=SEED)


def catboost_model():
    return catboost.CatBoostClassifier(iterations=1000, depth=6, loss_function="MultiClass", random_seed=SEED,
                                       verbose=0, allow_writing_files=False)

### Tune LightGBM and XGBoost on the validation year
Google's claim is that TabFM matches **tuned** gradient boosting, so the boosted trees get a fair chance: a random
search over the usual settings, scored by 2024 log loss, with early stopping. This makes their 2024 scores slightly
optimistic; section 8 checks every model on 2025, which nothing was tuned on.

In [ ]:
LGB_DEFAULT = dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8,
                   subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
XGB_DEFAULT = dict(n_estimators=400, learning_rate=0.03, max_depth=4, min_child_weight=1, subsample=0.8,
                   colsample_bytree=0.8, reg_lambda=1.0)


def lightgbm_model(params):
    return lgb.LGBMClassifier(objective="multiclass", random_state=SEED, n_jobs=-1, verbose=-1, **params)


def xgboost_model(params):
    return xgb.XGBClassifier(objective="multi:softprob", tree_method="hist", device="cuda" if HAS_GPU else "cpu",
                             random_state=SEED, **params)


def random_search(kind, X_tr, y_tr, X_ev, y_ev, n_trials):
    rng = np.random.default_rng(SEED)
    trials = []
    for i in range(n_trials):
        p = dict(learning_rate=float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.1])),
                 subsample=float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
                 colsample_bytree=float(rng.choice([0.5, 0.7, 0.8, 1.0])),
                 reg_lambda=float(rng.choice([0.0, 1.0, 5.0, 10.0])), n_estimators=3000)
        if kind == "lightgbm":
            p.update(num_leaves=int(rng.choice([4, 7, 15, 31, 63])), subsample_freq=1,
                     min_child_samples=int(rng.choice([10, 20, 50, 100, 200])))
            m = lightgbm_model(p)
            m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], callbacks=[lgb.early_stopping(100, verbose=False)])
            best_iter = m.best_iteration_
        else:
            p.update(max_depth=int(rng.choice([2, 3, 4, 6, 8])), min_child_weight=float(rng.choice([1, 5, 10, 20])))
            m = xgb.XGBClassifier(objective="multi:softprob", tree_method="hist",
                                  device="cuda" if HAS_GPU else "cpu", random_state=SEED,
                                  early_stopping_rounds=100, eval_metric="mlogloss", **p)
            m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], verbose=False)
            best_iter = m.best_iteration + 1
        loss = log_loss(y_ev, align_proba(m.classes_, m.predict_proba(X_ev)), labels=CLASSES)
        trials.append({**p, "n_estimators": max(int(best_iter), 20), "val_log_loss": loss})
    trials = pd.DataFrame(trials).sort_values("val_log_loss").reset_index(drop=True)
    best = trials.iloc[0].drop("val_log_loss").to_dict()
    int_keys = ["n_estimators", "num_leaves", "min_child_samples", "max_depth", "subsample_freq"]
    best = {k: (int(v) if k in int_keys else float(v)) for k, v in best.items() if pd.notna(v)}
    return best, trials


X_tr, y_tr = train[FEATURES], train["y_class"].values
X_va, y_va = val[FEATURES], val["y_class"].values

if TUNE_TREES:
    t0 = time.time()
    LGB_PARAMS, lgb_trials = random_search("lightgbm", X_tr, y_tr, X_va, y_va, N_TRIALS)
    XGB_PARAMS, xgb_trials = random_search("xgboost", X_tr, y_tr, X_va, y_va, N_TRIALS)
    print(f"Tuning took {time.time() - t0:.0f} s")
    print("Best LightGBM:", LGB_PARAMS, "| val log loss", round(lgb_trials['val_log_loss'].iloc[0], 4))
    print("Best XGBoost: ", XGB_PARAMS, "| val log loss", round(xgb_trials['val_log_loss'].iloc[0], 4))
else:
    LGB_PARAMS, XGB_PARAMS = LGB_DEFAULT, XGB_DEFAULT
    print("Tuning off: using the default LightGBM / XGBoost settings")

In [ ]:
def model_zoo(last_col, lgb_params, xgb_params, tabfm=True):
    # name -> a function that builds a fresh, unfitted model
    zoo = {
        "Majority class": lambda: Majority(),
        "Persistence": lambda: Persistence(last_col),
        "Logistic regression": logistic,
        "Random forest": random_forest,
        "HistGradientBoosting": hist_gb,
        "LightGBM" + (" (tuned)" if TUNE_TREES else ""): lambda: lightgbm_model(lgb_params),
        "XGBoost" + (" (tuned)" if TUNE_TREES else ""): lambda: xgboost_model(xgb_params),
    }
    if catboost is not None:
        zoo["CatBoost"] = catboost_model
    if tabfm and HAS_TABFM and RUN_TABFM:
        zoo["TabFM"] = lambda: TabFMWrapper("default")
    if tabfm and HAS_TABFM and RUN_TABFM_ENSEMBLE:
        zoo["TabFM ensemble"] = lambda: TabFMWrapper("ensemble")
    return zoo


def run_models(zoo, X_fit, y_fit, X_eval):
    # Fits every model, returns {name: proba on X_eval} and a timing table. A failing model is reported and skipped.
    probas, times = {}, []
    for name, build in zoo.items():
        t0 = time.time()
        try:
            model = build()
            model.fit(X_fit, y_fit)
            t1 = time.time()
            probas[name] = align_proba(model.classes_, model.predict_proba(X_eval))
            t2 = time.time()
            times.append({"model": name, "fit_s": t1 - t0, "predict_s": t2 - t1})
            print(f"{name:24s} fit {t1 - t0:7.1f} s | predict {t2 - t1:7.1f} s")
        except Exception:
            print(f"{name:24s} FAILED, skipped:")
            traceback.print_exc()
    return probas, pd.DataFrame(times).set_index("model")

## 3. Validation benchmark: train ≤ 2023 → predict 2024

In [ ]:
# Persistence needs the last-class column even when it is not one of the features
fit_cols = list(dict.fromkeys(FEATURES + [LAST_CLASS]))
zoo = model_zoo(LAST_CLASS, LGB_PARAMS, XGB_PARAMS)
PRIOR = class_prior(y_tr)


class Restrict:
    # Wraps a model so it only sees `cols` (keeps the zoo simple: every model gets the same frame)
    def __init__(self, model, cols):
        self.model, self.cols = model, cols

    def fit(self, X, y):
        self.model.fit(X[self.cols], y)
        self.classes_ = self.model.classes_
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X[self.cols])


def restricted(zoo, cols):
    return {name: (build if name == "Persistence" else (lambda b=build: Restrict(b(), cols)))
            for name, build in zoo.items()}


val_probas, val_times = run_models(restricted(zoo, FEATURES), train[fit_cols], y_tr, val[fit_cols])

In [ ]:
def leaderboard(probas, y, last_class, prior, times=None):
    board = pd.DataFrame({m: score(y, p, last_class, prior) for m, p in probas.items()}).T
    ranks = pd.DataFrame({k: board[k].rank(ascending=not HIGHER_IS_BETTER[k]) for k in RANK_METRICS})
    board.insert(0, "mean_rank", ranks.mean(axis=1))
    if times is not None:
        board = board.join(times)
    return board.sort_values(["mean_rank", "macro_f1"], ascending=[True, False])


def show_board(board, title):
    display(Markdown(f"#### {title}"))
    fmt = {c: "{:.3f}" for c in board.columns}
    fmt.update({"mean_rank": "{:.1f}", "missed_blooms": "{:.0f}", "false_alarms": "{:.0f}",
                "fit_s": "{:.1f}", "predict_s": "{:.1f}"})
    styler = board.style.format(fmt, na_rep="-")
    for col, up in HIGHER_IS_BETTER.items():
        if col in board.columns and board[col].notna().any():
            styler = styler.background_gradient(subset=[col], cmap="Greens" if up else "Greens_r")
    styler = styler.background_gradient(subset=["mean_rank"], cmap="Blues_r")
    display(styler)


val_board = leaderboard(val_probas, y_va, val[LAST_CLASS], PRIOR, val_times)
show_board(val_board, f"Validation 2024 · site-week table · features: {FEATURE_SET} · rule: {DECISION_RULE}")
val_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_val2024_site_{FEATURE_SET}.csv"))
print("mean_rank = average rank over", RANK_METRICS, "(1 = best). Greener = better in each column.")

In [ ]:
def is_tabfm(name):
    return name.startswith("TabFM")


def color_of(name):
    if is_tabfm(name):
        return "#d62728"
    if name in ("Majority class", "Persistence"):
        return "#7f7f7f"
    return "#1f77b4"


def metric_bars(board, metrics, title):
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 0.42 * len(board) + 1.5), sharey=True)
    order = board.sort_values("macro_f1").index
    for ax, m in zip(np.atleast_1d(axes), metrics):
        vals = board.loc[order, m]
        ax.barh(order, vals, color=[color_of(n) for n in order])
        if "Persistence" in board.index:
            ax.axvline(board.loc["Persistence", m], color="k", ls="--", lw=1)
        for i, v in enumerate(vals):
            if pd.notna(v):
                ax.text(v, i, f" {v:.3f}", va="center", fontsize=8)
        ax.set_title(m + (" (higher = better)" if HIGHER_IS_BETTER[m] else " (lower = better)"), fontsize=10)
        ax.set_xlim(0, max(1.0, np.nanmax(vals.values) * 1.15) if HIGHER_IS_BETTER[m] else None)
    fig.suptitle(title + "   (red = TabFM, blue = trained models, grey = baselines, dashed = persistence)", y=1.02)
    plt.tight_layout()
    plt.show()


metric_bars(val_board, ["macro_f1", "auc_any", "onset_recall", "onset_auc"], "Validation 2024")

In [ ]:
def confusion_grid(probas, y, prior, title):
    names = list(probas)
    ncol = 4
    nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow))
    for ax, name in zip(axes.ravel(), names):
        cm = confusion_matrix(y, decide(probas[name], prior), labels=CLASSES)
        share = cm / cm.sum(1, keepdims=True).clip(1)
        labels = np.array([[f"{share[i, j]:.0%}\n({cm[i, j]})" for j in range(3)] for i in range(3)])
        sns.heatmap(share, annot=labels, fmt="", cmap="Reds" if is_tabfm(name) else "Blues", vmin=0, vmax=1,
                    cbar=False, ax=ax, xticklabels=["pred 0", "pred 1", "pred 2"], yticklabels=["true 0", "true 1", "true 2"])
        ax.set_title(name, fontsize=10)
    for ax in axes.ravel()[len(names):]:
        ax.axis("off")
    fig.suptitle(title + " · row % (count)", y=1.0)
    plt.tight_layout()
    plt.show()


confusion_grid(val_probas, y_va, PRIOR, "Confusion matrices, validation 2024")

In [ ]:
def roc_pr_curves(probas, y, title):
    pos = np.asarray(y) > 0
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    for name, p in probas.items():
        if name == "Majority class":
            continue
        s = p[:, 1] + p[:, 2]
        fpr, tpr, _ = roc_curve(pos, s)
        prec, rec, _ = precision_recall_curve(pos, s)
        style = dict(lw=2.4 if is_tabfm(name) else 1.3, ls="-" if is_tabfm(name) else ("--" if name == "Persistence" else "-"))
        axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(pos, s):.3f})", **style)
        axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(pos, s):.3f})", **style)
    axes[0].plot([0, 1], [0, 1], color="grey", lw=0.8)
    axes[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC: next week ≥ 5,000 cells/L")
    axes[1].axhline(pos.mean(), color="grey", lw=0.8)
    axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall: next week ≥ 5,000 cells/L")
    for ax in axes:
        ax.legend(fontsize=8, loc="lower right" if ax is axes[0] else "upper right")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


roc_pr_curves(val_probas, y_va, "Validation 2024 (thick lines = TabFM)")

### Are the differences real? Week-block bootstrap
Each dot is a model's score; the bar is its 95 % interval when the 2024 weeks are resampled. The table compares each
TabFM variant with the best trained tree model **on the same resamples** (paired), which is tighter than comparing the
two intervals by eye.

In [ ]:
def bootstrap_report(probas, y, weeks, last_class, prior, title):
    names = [m for m in probas if m != "Majority class"]
    boots = week_bootstrap(y, {m: probas[m] for m in names}, weeks, last_class, prior)
    fig, axes = plt.subplots(1, len(boots), figsize=(5.2 * len(boots), 0.4 * len(names) + 1.6), sharey=True)
    point = pd.DataFrame({m: score(y, probas[m], last_class, prior) for m in names}).T
    order = point.sort_values("macro_f1").index
    for ax, (k, b) in zip(axes, boots.items()):
        lo, hi = b.quantile(0.025), b.quantile(0.975)
        for i, m in enumerate(order):
            ax.plot([lo[m], hi[m]], [i, i], color=color_of(m), lw=2)
            ax.plot(point.loc[m, k], i, "o", color=color_of(m))
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(order)
        ax.set_title(f"{k} with 95% interval", fontsize=10)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

    trees = [m for m in names if not is_tabfm(m) and m not in ("Persistence",)]
    rows = []
    for tab in [m for m in names if is_tabfm(m)]:
        for k, b in boots.items():
            best_tree = point.loc[trees, k].idxmax()
            diff = (b[tab] - b[best_tree]).dropna()
            rows.append({"TabFM variant": tab, "metric": k, "best trained model": best_tree,
                         "TabFM": point.loc[tab, k], "best trained": point.loc[best_tree, k],
                         "difference": point.loc[tab, k] - point.loc[best_tree, k],
                         "95% CI low": diff.quantile(0.025), "95% CI high": diff.quantile(0.975),
                         "P(TabFM better)": (diff > 0).mean()})
    if rows:
        table = pd.DataFrame(rows)
        display(table.style.format({c: "{:.3f}" for c in table.columns if table[c].dtype.kind == "f"}))
        return boots, table
    print("No TabFM rows to compare (TabFM switched off or failed).")
    return boots, None


val_boots, val_tabfm_vs_best = bootstrap_report(val_probas, y_va, val["week_start"].values, val[LAST_CLASS].values,
                                                PRIOR, "Validation 2024, week-block bootstrap")

In [ ]:
def calibration_plot(probas, y, title, n_bins=10):
    pos = np.asarray(y) > 0
    keep = [m for m in probas if m not in ("Majority class", "Persistence")]
    fig, ax = plt.subplots(figsize=(6.5, 6))
    for name in keep:
        s = probas[name][:, 1] + probas[name][:, 2]
        bins = np.quantile(s, np.linspace(0, 1, n_bins + 1))
        idx = np.clip(np.digitize(s, bins[1:-1]), 0, n_bins - 1)
        df = pd.DataFrame({"s": s, "y": pos, "b": idx}).groupby("b").mean()
        ax.plot(df["s"], df["y"], "o-", lw=2.4 if is_tabfm(name) else 1.2, ms=4, label=name)
    ax.plot([0, 1], [0, 1], color="grey", lw=0.8)
    ax.set(xlabel="predicted P(≥ 5k next week)", ylabel="observed share ≥ 5k", title=title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


calibration_plot(val_probas, y_va, "Calibration, validation 2024 (closer to the diagonal = more honest probabilities)")

In [ ]:
def timeline(df, probas, y_col, title, models=None):
    # Weekly mean predicted P(>= 5k) across sites vs the observed share of sites >= 5k
    if models is None:
        trained = [m for m in probas if not is_tabfm(m) and m not in ("Majority class", "Persistence")]
        best_trained = max(trained, key=lambda m: safe_auc(df[y_col].values > 0, probas[m][:, 1] + probas[m][:, 2]))
        models = ["Persistence", best_trained] + [m for m in probas if is_tabfm(m)]
    weekly = pd.DataFrame({"week": df["target_week"].values, "observed": (df[y_col].values > 0).astype(float)})
    for m in models:
        weekly[m] = probas[m][:, 1] + probas[m][:, 2]
    weekly = weekly.groupby("week").mean()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(weekly.index, weekly["observed"], step="mid", color="#fdae6b", alpha=0.6, label="observed share ≥ 5k")
    for m in models:
        ax.plot(weekly.index, weekly[m], lw=2.2 if is_tabfm(m) else 1.3, label=m)
    ax.set(ylabel="share of sites / mean probability", title=title, ylim=(0, 1))
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout()
    plt.show()


timeline(val, val_probas, "y_class", "Forecast week by week, 2024: mean predicted risk across sites vs what happened")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.4 * len(val_times) + 1.2))
t = (val_times["fit_s"] + val_times["predict_s"]).sort_values()
ax.barh(t.index, t.values, color=[color_of(n) for n in t.index])
ax.set_xscale("log")
ax.set(xlabel="seconds (fit + predict, log scale)", title="Run time on this machine")
for i, v in enumerate(t.values):
    ax.text(v, i, f" {v:.1f}s", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Learning curve: does TabFM win with less data?
Each model is trained on a random sample of the training rows (2 seeds) and scored on all of 2024. Foundation models
usually lead at small sizes; the question for us is whether trained models catch up by our full 12,379 rows.
TabFM uses 8 ensemble members here to keep the run short, so its curve is a little below the full model.

In [ ]:
if RUN_LEARNING_CURVE:
    sizes = [250, 500, 1000, 2000, 4000, 8000, len(train)]
    # LightGBM uses the default settings here: the tuned ones (e.g. min_child_samples) assume the full training set
    lc_zoo = {"Logistic regression": logistic, "Random forest": random_forest,
              "LightGBM (default)": lambda: lightgbm_model(LGB_DEFAULT)}
    if HAS_TABFM and RUN_TABFM:
        lc_zoo["TabFM (8 members)"] = lambda: TabFMWrapper("default", n_estimators=8)
    rows = []
    for n in sizes:
        for seed in ([0, 1] if n < len(train) else [0]):
            sub = train.sample(n=n, random_state=seed) if n < len(train) else train
            for name, build in lc_zoo.items():
                try:
                    m = build().fit(sub[FEATURES], sub["y_class"].values)
                    p = align_proba(m.classes_, m.predict_proba(val[FEATURES]))
                    s = score(y_va, p, val[LAST_CLASS], class_prior(sub["y_class"]))
                    rows.append({"n_train": n, "seed": seed, "model": name, **s})
                except Exception as e:
                    print(f"{name} at n={n} failed: {e}")
        print(f"n = {n:>6,} done")
    lc = pd.DataFrame(rows)
    lc.to_csv(os.path.join(OUTPUT_DIR, "learning_curve.csv"), index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for ax, metric in zip(axes, ["auc_any", "macro_f1", "onset_auc"]):
        for name, g in lc.groupby("model"):
            g = g.groupby("n_train")[metric].mean()
            ax.plot(g.index, g.values, "o-", lw=2.4 if is_tabfm(name) else 1.4, label=name,
                    color="#d62728" if is_tabfm(name) else None)
        ax.axhline(val_board.loc["Persistence", metric], color="k", ls="--", lw=1, label="Persistence")
        ax.set_xscale("log")
        ax.set(xlabel="training rows (log scale)", title=f"{metric} on 2024")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Feature-set sweep
Notebook 05 found that the environmental groups did not help simple models. TabFM's pitch is that it finds feature
interactions without hand-made features, so this checks whether it uses them better. Chlorophyll starts in 2018 and
gridded wind in 2019, so those groups are missing for many training rows.

In [ ]:
if RUN_FEATURE_SWEEP:
    sweep_sets = ["core", "core+temperature", "core+chlorophyll", "core+wind", "core+all"]
    sw_zoo = {"Logistic regression": logistic, "Random forest": random_forest,
              "LightGBM": lambda: lightgbm_model(LGB_PARAMS)}
    if HAS_TABFM and RUN_TABFM:
        sw_zoo["TabFM (8 members)"] = lambda: TabFMWrapper("default", n_estimators=8)
    rows = []
    for fs_name in sweep_sets:
        cols = site_features(fs_name)
        for name, build in sw_zoo.items():
            try:
                m = build().fit(train[cols], y_tr)
                p = align_proba(m.classes_, m.predict_proba(val[cols]))
                rows.append({"feature_set": fs_name, "model": name, **score(y_va, p, val[LAST_CLASS], PRIOR)})
            except Exception as e:
                print(f"{name} on {fs_name} failed: {e}")
        print(f"{fs_name:18s} done ({len(cols)} features)")
    sweep = pd.DataFrame(rows)
    sweep.to_csv(os.path.join(OUTPUT_DIR, "feature_sweep.csv"), index=False)

    fig, axes = plt.subplots(1, 3, figsize=(19, 3.6))
    for ax, metric in zip(axes, ["auc_any", "macro_f1", "onset_auc"]):
        pivot = sweep.pivot(index="model", columns="feature_set", values=metric)[sweep_sets]
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="Greens", ax=ax, cbar=False)
        ax.set_title(f"{metric} on 2024")
        ax.set(xlabel="", ylabel="")
        ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()

## 6. Which features drive the predictions?
Permutation importance: shuffle one feature in 2024 and measure how much **AUC ≥5k** drops (5 repeats). It works the
same way for any model, including TabFM (switch on `RUN_TABFM_IMPORTANCE`; it needs one TabFM prediction per feature
and repeat).

In [ ]:
def permutation_importance_auc(model, X, y, n_repeats=5):
    rng = np.random.default_rng(SEED)
    pos = np.asarray(y) > 0
    base = roc_auc_score(pos, align_proba(model.classes_, model.predict_proba(X))[:, 1:].sum(1))
    out = {}
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].values)
            drops.append(base - roc_auc_score(pos, align_proba(model.classes_, model.predict_proba(Xp))[:, 1:].sum(1)))
        out[col] = np.mean(drops)
    return pd.Series(out).sort_values()


if RUN_IMPORTANCE:
    imp_models = {"Random forest": random_forest(), "LightGBM": lightgbm_model(LGB_PARAMS), "Logistic regression": logistic()}
    if HAS_TABFM and RUN_TABFM and RUN_TABFM_IMPORTANCE:
        imp_models["TabFM (8 members)"] = TabFMWrapper("default", n_estimators=8, cache_context=True)
    importances = {}
    for name, m in imp_models.items():
        t0 = time.time()
        m.fit(train[FEATURES], y_tr)
        importances[name] = permutation_importance_auc(m, val[FEATURES], y_va, n_repeats=2 if is_tabfm(name) else 5)
        print(f"{name}: {time.time() - t0:.0f} s")
    imp = pd.DataFrame(importances)
    imp.to_csv(os.path.join(OUTPUT_DIR, f"permutation_importance_{FEATURE_SET}.csv"))
    order = imp.mean(axis=1).sort_values().index
    imp.loc[order].plot.barh(figsize=(10, 0.35 * len(order) + 1.5), width=0.8)
    plt.axvline(0, color="k", lw=0.8)
    plt.xlabel("drop in 2024 AUC ≥5k when the feature is shuffled")
    plt.title("Permutation importance (bigger = the model relies on it more)")
    plt.tight_layout()
    plt.show()

## 7. Regional weekly table (364 training weeks)
One row per week for the whole focus coast (`regional_core`, persistence = `last_class`). This is a small table, the
size where foundation models are strongest. 2024 has only 52 labelled weeks (5 at risk, 9 bloom), so read these
scores as a rough check, not a ranking. The trees use fixed settings here (no tuning on 52 weeks).

In [ ]:
if RUN_REGIONAL:
    R_FEATURES = list(feature_sets["regional_core"])
    r_train = regional[regional["split"] == "train"]
    r_val = regional[regional["split"] == "validate"]
    r_cols = list(dict.fromkeys(R_FEATURES + ["last_class"]))
    r_zoo = model_zoo("last_class", dict(LGB_DEFAULT, num_leaves=7, min_child_samples=10), XGB_DEFAULT)
    r_zoo = {k.replace(" (tuned)", ""): v for k, v in r_zoo.items()}
    r_prior = class_prior(r_train["y_class"])
    r_probas, r_times = run_models(restricted(r_zoo, R_FEATURES), r_train[r_cols], r_train["y_class"].values,
                                   r_val[r_cols])
    r_board = leaderboard(r_probas, r_val["y_class"].values, r_val["last_class"], r_prior, r_times)
    show_board(r_board, f"Validation 2024 · regional table · {len(r_train)} training weeks, {len(r_val)} validation weeks")
    r_board.to_csv(os.path.join(OUTPUT_DIR, "leaderboard_val2024_regional.csv"))
    metric_bars(r_board, ["macro_f1", "auc_any", "ap_any"], "Regional table, validation 2024")

## 8. Final check: train ≤ 2024 → predict 2025, and false alarms in 2026
Every model is refitted on 2015–2024 with the settings chosen above (nothing is tuned on 2025) and predicts 2025.
2025 has only 49 at-risk and 28 bloom rows and 33 onsets, so use the intervals. The 2026 holdout has no blooms at
all, so it only counts false alarms.

In [ ]:
if SCORE_TEST:
    trval = site[site["split"].isin(["train", "validate"])]
    test = site[site["split"] == "test"]
    holdout = site[site["split"] == "holdout"]
    y_trval, y_te = trval["y_class"].values, test["y_class"].values
    prior_tv = class_prior(y_trval)
    eval_rows = pd.concat([test, holdout])

    # Boosted trees keep their tuned settings, with ~20 % more trees for the extra year of data
    lgb_final = dict(LGB_PARAMS, n_estimators=int(LGB_PARAMS["n_estimators"] * 1.2))
    xgb_final = dict(XGB_PARAMS, n_estimators=int(XGB_PARAMS["n_estimators"] * 1.2))
    final_probas_all, test_times = run_models(restricted(model_zoo(LAST_CLASS, lgb_final, xgb_final), FEATURES),
                                              trval[fit_cols], y_trval, eval_rows[fit_cols])
    test_probas = {m: p[:len(test)] for m, p in final_probas_all.items()}
    hold_probas = {m: p[len(test):] for m, p in final_probas_all.items()}

    test_board = leaderboard(test_probas, y_te, test[LAST_CLASS], prior_tv, test_times)
    test_board["false_alarms_2026"] = pd.Series({m: int((decide(p, prior_tv) > 0).sum()) for m, p in hold_probas.items()})
    show_board(test_board, f"TEST 2025 · trained on 2015–2024 · features: {FEATURE_SET} · rule: {DECISION_RULE}")
    test_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_test2025_site_{FEATURE_SET}.csv"))
    print(f"false_alarms_2026 = 2026 holdout rows (all truly class 0, n = {len(holdout)}) predicted as 1 or 2")

In [ ]:
if SCORE_TEST:
    metric_bars(test_board, ["macro_f1", "auc_any", "onset_recall", "onset_auc"], "Test 2025")
    confusion_grid(test_probas, y_te, prior_tv, "Confusion matrices, test 2025")
    roc_pr_curves(test_probas, y_te, "Test 2025 (thick lines = TabFM)")
    test_boots, test_tabfm_vs_best = bootstrap_report(test_probas, y_te, test["week_start"].values,
                                                      test[LAST_CLASS].values, prior_tv, "Test 2025, week-block bootstrap")
    timeline(test, test_probas, "y_class", "Forecast week by week, 2025")

## 9. Verdict
Printed from the tables above. "Clearly better" means the paired 95 % interval of the difference excludes zero.

In [ ]:
def verdict(board, comparison, label):
    lines = [f"### {label}"]
    trained = [m for m in board.index if m not in ("Majority class", "Persistence")]
    best_overall = board.index[0]
    lines.append(f"- Best mean rank: **{best_overall}** (mean rank {board.loc[best_overall, 'mean_rank']:.1f}).")
    for k in ["macro_f1", "auc_any", "onset_auc", "recall_2"]:
        col = board[k]
        best = col.idxmax()
        lines.append(f"- Best {k}: **{best}** ({col[best]:.3f}); persistence {board.loc['Persistence', k]:.3f}.")
    beats = [m for m in trained if board.loc[m, "macro_f1"] > board.loc["Persistence", "macro_f1"]]
    lines.append(f"- Models with a higher macro-F1 than persistence: {', '.join(beats) if beats else 'none'}.")
    if comparison is not None:
        for _, r in comparison.iterrows():
            if r["95% CI low"] > 0:
                call = "clearly better"
            elif r["95% CI high"] < 0:
                call = "clearly worse"
            else:
                call = "not distinguishable"
            lines.append(f"- {r['TabFM variant']} vs {r['best trained model']} on {r['metric']}: "
                         f"{r['difference']:+.3f} [{r['95% CI low']:+.3f}, {r['95% CI high']:+.3f}] → **{call}**.")
    else:
        lines.append("- TabFM was not run, so there is no TabFM comparison.")
    display(Markdown("\n".join(lines)))


verdict(val_board, val_tabfm_vs_best, "Validation 2024")
if SCORE_TEST:
    verdict(test_board, test_tabfm_vs_best, "Test 2025")

### How to read the result
- **TabFM clearly better on both 2024 and 2025:** worth carrying forward as a candidate. Keep a tree model next to it
  for the driver analysis, because TabFM has no coefficients, splits or SHAP values, only permutation importance.
- **Not distinguishable:** prefer the tree model. It explains its drivers, runs in seconds without a GPU, and has no
  licence restriction. The default TabFM weights are licensed for **non-commercial use only**, so Biointerphase
  could not use them in a product.
- **Clearly worse:** report it as a tested and rejected approach. That is still a useful result for the write-up.

In every case, the gap to **persistence** and the **onset** scores matter more than small gaps between models. Most
of the skill comes from bloom history, so the data limits the forecast more than the choice of learner does.